# Houston Zillow LLM Automated Valuation Model

---
| Section | Content |
|---|---|
| 1 | Setup and project config |
| 2 | Data collection |
| 3 | Data cleaning and sample construction |
| 4 | Descriptive statistics |
| 5 | Baseline hedonic regressions |
| 6 | LLM feature extraction pipeline |
| 7 | Manual audit (20 entries) |
| 8 | Augmented models |
| 9 | Interpretation and diagnostics |
| 10 | Save outputs |

---
## 1 · Setup and Project Configuration

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

# Add project root to path (works whether notebook is run from root or notebooks/)
root = Path().absolute()
if root.name == 'notebooks':
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Project root: {root}')

In [ ]:
from src.config import (
    DATA_RAW, DATA_INTERIM, DATA_PROCESSED,
    OUTPUTS_FIGS, OUTPUTS_TABLES,
    PRICE_MIN, PRICE_MAX, DAYS_ON_ZILLOW, TARGET_N,
    HOUSTON_ZIP_CODES, LLM_MODEL,
    RANDOM_SEED, APIFY_API_TOKEN, OPENAI_API_KEY,
)
from src.utils import ensure_dirs

ensure_dirs(DATA_RAW, DATA_INTERIM, DATA_PROCESSED, OUTPUTS_FIGS, OUTPUTS_TABLES)

# Quick credential check (values are masked)
config_summary = pd.Series({
    'Target listings':   TARGET_N,
    'Price range':       f'${PRICE_MIN:,} – ${PRICE_MAX:,}',
    'Days on Zillow':    f'≤ {DAYS_ON_ZILLOW}',
    'ZIP codes covered': len(HOUSTON_ZIP_CODES),
    'LLM model':         LLM_MODEL,
    'APIFY_API_TOKEN':   '✓ set' if APIFY_API_TOKEN else '✗ MISSING',
    'OPENAI_API_KEY':    '✓ set' if OPENAI_API_KEY  else '✗ MISSING',
}, name='value').to_frame()

display(config_summary)

---
## 2 · Data Collection

In [ ]:
from src.data_collection import collect_listings
from src.data_cleaning import normalize_detail_records

# Set force_refresh=True to re-scrape (spends Apify credits)
FORCE_REFRESH = False

zip_records, detail_records = collect_listings(force_refresh=FORCE_REFRESH)

print(f'ZIP search records  : {len(zip_records):,}')
print(f'Detail records      : {len(detail_records):,}')

In [ ]:
# Normalize detail records to a flat DataFrame
df_raw = normalize_detail_records(detail_records)

print(f'Raw DataFrame: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

# Persist raw table
df_raw.to_parquet(DATA_RAW / 'listings_raw.parquet', index=False)
df_raw.to_csv(DATA_RAW / 'listings_raw.csv', index=False)
print('Raw table saved to data/raw/')

display(df_raw.head(3))

### Takeaway: Data Collection


---
## 3 · Data Cleaning and Sample Construction

In [ ]:
from src.data_cleaning import clean_listings, summary_stats, missingness_table

df_clean, attrition = clean_listings(df_raw)

print(f'\nFinal sample: {len(df_clean):,} listings\n')
display(attrition.style.set_caption('Attrition table'))

In [ ]:
df_clean.to_parquet(DATA_INTERIM / 'listings_clean.parquet', index=False)
df_clean.to_csv(DATA_INTERIM / 'listings_clean.csv', index=False)
attrition.to_csv(OUTPUTS_TABLES / 'attrition.csv', index=False)
print('Cleaned table saved to data/interim/')

# Column overview
print(f'\nColumns: {list(df_clean.columns)}')

### Sample Construction Waterfall


In [ ]:
from src.plotting import plot_sample_waterfall

pipeline_steps = [
    ('ZIP search (raw)',             2500),
    ('Pre-filter: excl. home types', 2414),
    ('Capped to scrape target',      1800),
    ('Detail scraper returned',      len(df_raw)),
]
for _, row in attrition.iterrows():
    if row['n_dropped'] > 0:
        pipeline_steps.append((f"Clean: {row['step']}", int(row['n_remaining'])))
pipeline_steps.append(('With description (LLM-ready)',
                        int(df_clean['has_description'].sum())))

fig = plot_sample_waterfall(pipeline_steps)
plt.show()


### Variable Catalog - All Scraped Columns


In [ ]:
from src.data_cleaning import build_variable_catalog

catalog = build_variable_catalog(df_raw)
catalog.to_csv(OUTPUTS_TABLES / 'variable_catalog.csv', index=False)
print(f'Variable catalog: {len(catalog)} columns across {catalog.category.nunique()} categories')

def _cov_color(val):
    if not isinstance(val, (int, float)): return ''
    if val >= 80: return 'background-color: #d4edda'
    if val >= 50: return 'background-color: #fff3cd'
    return 'background-color: #f8d7da'

display(
    catalog.style
        .map(_cov_color, subset=['pct_coverage'])
        .set_caption('Variable Catalog  |  green >= 80% coverage  |  yellow >= 50%  |  red < 50%')
        .set_properties(**{'font-size': '11px'})
        .hide(axis='index')
)


---
## 4 · Descriptive Statistics

In [ ]:
stats = summary_stats(df_clean)
display(stats.style.set_caption('Summary statistics — key numeric variables'))
stats.to_csv(OUTPUTS_TABLES / 'summary_stats.csv')

## Spatial Analysis: Houston Market Geography in 3D

| Chart | Height | Colour | Resolution |
|---|---|---|---|
| A | Listing count | Median price | Hex grid |
| B-hex | Median $/sqft | Median $/sqft | Hex grid |
| B-zip | Median $/sqft | Median $/sqft | ZIP column |
| C | Avg LLM luxury score | Avg LLM luxury score | Hex grid |

In [ ]:
from src.plotting_3d import (
    chart_a_density_price,
    chart_b_ppsf_hex,
    chart_b_ppsf_zip,
    chart_c_luxury,
    save_deck,
    show_deck,
)

# Chart A — listing density + median price (hex grid)
deck_a = chart_a_density_price(df_clean)
save_deck(deck_a, OUTPUTS_FIGS / '3d_a_density_price.html')
show_deck(deck_a)

In [ ]:
# Chart B — $/sqft hex grid
deck_b_hex = chart_b_ppsf_hex(df_clean)
save_deck(deck_b_hex, OUTPUTS_FIGS / '3d_b_ppsf_hex.html')
show_deck(deck_b_hex)

In [ ]:
# Chart B — $/sqft ZIP column version
deck_b_zip = chart_b_ppsf_zip(df_clean)
save_deck(deck_b_zip, OUTPUTS_FIGS / '3d_b_ppsf_zip.html')
show_deck(deck_b_zip)

In [ ]:
# Load df_llm from cache if not already in memory (e.g. after partial kernel restart)
try:
    df_llm
except NameError:
    df_llm = pd.read_parquet(DATA_PROCESSED / 'listings_with_llm.parquet')
    print(f"Loaded df_llm from cache: {df_llm.shape[0]:,} rows")

In [ ]:
# Chart C — LLM luxury score geography (requires LLM pipeline to have run)
deck_c = chart_c_luxury(df_llm)
save_deck(deck_c, OUTPUTS_FIGS / '3d_c_luxury_score.html')
show_deck(deck_c)

In [ ]:
miss = missingness_table(df_clean)
print('Columns with any missing data:')
display(miss[miss['n_missing'] > 0])

In [ ]:
from src.plotting import (
    plot_price_histogram, plot_sqft_histogram, plot_sqft_vs_price,
    plot_price_by_home_type, plot_listings_by_zip,
    plot_correlation_heatmap, plot_map,
)

_ = plot_price_histogram(df_clean)
_ = plot_sqft_histogram(df_clean)
_ = plot_sqft_vs_price(df_clean)
_ = plot_price_by_home_type(df_clean)
_ = plot_listings_by_zip(df_clean)
_ = plot_correlation_heatmap(df_clean)
plot_map(df_clean)   # saves HTML + PNG; returns None
plt.show()
print('All charts saved to outputs/figures/')

In [ ]:
from src.plotting import (
    plot_price_by_bedrooms, plot_price_per_sqft,
    plot_year_built_vs_price, plot_days_on_market,
    plot_zip_price_bubbles,
)

_ = plot_price_by_bedrooms(df_clean);    plt.show()
_ = plot_price_per_sqft(df_clean);       plt.show()
_ = plot_year_built_vs_price(df_clean);  plt.show()
_ = plot_days_on_market(df_clean);       plt.show()
_ = plot_zip_price_bubbles(df_clean);    plt.show()
print('All charts saved to outputs/figures/')


### Additional Descriptive Charts

In [ ]:
# code cell id: desc-extra-charts
from src.plotting import plot_desc_quality_vs_price, plot_age_vs_ppsf, plot_desc_length_dist

plot_age_vs_ppsf(df_clean, save_path=OUTPUTS_FIGS / '15_age_vs_ppsf.png')
plot_desc_length_dist(df_clean, save_path=OUTPUTS_FIGS / '16_desc_length_dist.png')
plt.show()

In [ ]:
from src.plotting_3d import chart_e_distress, save_deck, show_deck
deck_e = chart_e_distress(df_llm)
save_deck(deck_e, OUTPUTS_FIGS / '3d_e_distress.html')
show_deck(deck_e)

### Geographic Distribution — Density & Price Maps



In [ ]:
from src.plotting import plot_map, plot_density_map, plot_hexbin_map

# Interactive scatter map (price per listing) — saved as HTML
plot_map(df_clean)

# Interactive density heatmap (listing concentration) — saved as HTML
plot_density_map(df_clean)

# Static side-by-side hexbin: count left, median price right
fig = plot_hexbin_map(df_clean)
plt.show()

print('Maps saved to outputs/figures/')
print('  07_scatter_map.html  — open in browser for interactive price map')
print('  15_density_map.html  — open in browser for interactive density map')
print('  16_hexbin_map.png    — static hexbin (count + price) for papers')


### Home-type and ZIP distribution

In [ ]:
print('Home type counts:')
display(df_clean['home_type'].value_counts().to_frame('count'))

print('\nTop 15 ZIP codes by listing count:')
display(df_clean['zip_code'].value_counts().head(15).to_frame('count'))

---
## 5 · Baseline Hedonic Regression



In [ ]:
from src.features_structured import build_feature_matrix, get_target
from src.modeling import (
    fit_ols, fit_xgboost, fit_lasso_cv,
    lasso_selected_features, shared_holdout_split,
    paired_rmse_test, build_two_panel_table, build_comparison_table,
)

y = get_target(df_clean)

# OLS-Structured: lean structured baseline
X_s = build_feature_matrix(df_clean, include_llm=False)

# Shared 80/20 hold-out split -- reused by every structured-only baseline model
X_s_train, X_s_test, y_train, y_test = shared_holdout_split(X_s, y)

print(f'Hold-out split: {len(X_s_train):,} train / {len(X_s_test):,} test')
print(f'OLS-Structured structured features: {X_s_train.shape[1]}')


In [ ]:
ols_s, metrics_s, coef_s = fit_ols(
    X_s_train, y_train,
    model_name='OLS-Structured',
    X_test=X_s_test, y_test=y_test,
)

# Display top non-FE coefficients for OLS-Structured
non_fe = coef_s[~coef_s.index.str.startswith('zip_code_')].copy()
print('OLS-Structured — Key coefficients (non-ZIP-FE):')
display(non_fe.sort_values('p_value').head(20))


In [ ]:
# Save coefficient table
coef_s.to_csv(OUTPUTS_TABLES / 'coef_ols_structured.csv')

display(build_comparison_table([metrics_s]).style.set_caption('Structured baseline performance'))


In [ ]:
# Cross-validation already embedded in fit_ols (run_cv=True)
# Show predictive fit for the structured baseline
from src.plotting import plot_actual_vs_predicted
import statsmodels.api as sm

X_s_c = sm.add_constant(X_s, has_constant='add')
y_pred_s = ols_s.predict(X_s_c).values

_ = plot_actual_vs_predicted(
    y.values,
    y_pred_s,
    model_name='OLS-Structured',
    save_name='ols_structured_actual_vs_predicted',
)
plt.show()


### Spatial Analysis — Price Patterns & Regression Diagnostics



In [ ]:
import statsmodels.api as sm
from src.plotting import (
    plot_price_gradient, plot_residual_map,
    plot_price_surface, plot_zip_fe_chart,
)

# 1. Monocentric price gradient
_ = plot_price_gradient(df_clean)
plt.show()

# 2. OLS-Structured residuals — spatial autocorrelation diagnostic
X_s_c = sm.add_constant(X_s, has_constant='add')
residuals = y.values - ols_s.predict(X_s_c).values
_ = plot_residual_map(df_clean, residuals, model_name='OLS-Structured')
plt.show()

# 3. Interpolated price surface (thin-plate spline)
_ = plot_price_surface(df_clean)
plt.show()

# 4. ZIP fixed-effect coefficients vs. median price
_ = plot_zip_fe_chart(df_clean, coef_s)
plt.show()


---
## 6 · LLM Feature Extraction Pipeline



In [ ]:
# Inspect description availability
n_with_desc = df_clean['has_description'].sum()
print(f'Listings with usable description: {n_with_desc:,} / {len(df_clean):,} '
      f'({n_with_desc/len(df_clean)*100:.1f}%)')

# Show a sample description
sample_desc = df_clean[df_clean['has_description']]['property_description'].iloc[0]
print(f'\nSample description ({len(sample_desc)} chars):')
print(sample_desc[:500] + ('...' if len(sample_desc) > 500 else ''))

In [ ]:
import json
from src.llm_schema import get_json_schema, SYSTEM_PROMPT

schema = get_json_schema()
print('System prompt:')
print(SYSTEM_PROMPT)
print('\nTop-level schema keys:', list(schema['schema'].get('properties', {}).keys()))

In [ ]:
from src.config import LLM_RUN_MODE, LLM_IMMEDIATE_MAX_WORKERS

print(f"LLM run mode        : {LLM_RUN_MODE}")
print(f"  batch   → OpenAI Batch API, 50% cost reduction, ≤24h completion")
print(f"  immediate → concurrent sync calls ({LLM_IMMEDIATE_MAX_WORKERS} workers), results in minutes")
print()
print("To switch modes edit src/config.py:")
print("  LLM_RUN_MODE = 'batch'      # cheap, async")
print("  LLM_RUN_MODE = 'immediate'  # fast, synchronous")

In [ ]:
from src.llm_batch import run_llm_pipeline

# Keep this False during normal notebook runs so cached LLM outputs are reused.
FORCE_REBUILD = False

df_llm, features_df, failures_df = run_llm_pipeline(df_clean, force_rebuild=FORCE_REBUILD)

if not failures_df.empty:
    print(f'\n{len(failures_df)} parse failures:')
    display(failures_df.head(10))


In [ ]:
# Optional single-call smoke test (disabled by default; requires live API access)
RUN_LLM_SMOKE_TEST = False

if RUN_LLM_SMOKE_TEST and OPENAI_API_KEY:
    from src.llm_batch import _client, _call_single
    from src.llm_schema import get_json_schema

    sample_row = df_clean[df_clean['has_description']].iloc[0]
    result = _call_single(
        _client(),
        listing_id=str(sample_row['zpid']),
        description=sample_row['property_description'],
        model=LLM_MODEL,
        schema=get_json_schema(),
    )
    print('ok:', result['ok'])
    if result['ok']:
        print('Sample output keys:', list(result['row'].keys())[:6], '...')
    else:
        print('ERROR:', result['error'])
else:
    print('Skipped live LLM smoke test; using cached pipeline outputs instead.')


In [ ]:
# Persist and inspect results
df_llm.to_parquet(DATA_PROCESSED / 'listings_with_llm.parquet', index=False)
df_llm.to_csv(DATA_PROCESSED / 'listings_with_llm.csv', index=False)
features_df.to_csv(OUTPUTS_TABLES / 'llm_features.csv')

llm_cols = [c for c in df_llm.columns if c.startswith('llm_')]
print(f'Merged DataFrame: {df_llm.shape[0]:,} rows × {df_llm.shape[1]} columns  '
      f'({len(llm_cols)} LLM columns)')

if llm_cols:
    display(df_llm[llm_cols].describe().T.head(15))
else:
    print("No LLM columns found — check for pipeline errors above.")

In [ ]:
from src.plotting import plot_llm_score_distributions

_ = plot_llm_score_distributions(df_llm)
plt.show()

In [ ]:
from src.features_structured import LLM_SCORE_COLS

llm_stats = df_llm[LLM_SCORE_COLS].describe().T
llm_stats['skew'] = df_llm[LLM_SCORE_COLS].skew()
llm_stats['kurtosis'] = df_llm[LLM_SCORE_COLS].kurtosis()
display(llm_stats.round(2))

### LLM Binary Flag Prevalence


In [ ]:
from src.features_structured import LLM_FLAG_COLS

flag_rates = df_llm[LLM_FLAG_COLS].mean().sort_values(ascending=False)
flag_df = pd.DataFrame({'flag': flag_rates.index, 'prevalence': flag_rates.values})
flag_df['prevalence_pct'] = (flag_df['prevalence'] * 100).round(1)
flag_df['flag'] = flag_df['flag'].str.replace('llm_', '')
display(flag_df.style.set_caption('LLM Flag Prevalence Rates'))

In [ ]:
from src.plotting import plot_desc_quality_vs_price, plot_llm_scores_vs_price, plot_llm_coverage_by_zip

plot_desc_quality_vs_price(df_llm, save_path=OUTPUTS_FIGS / '14_desc_quality_vs_price.png')
plot_llm_scores_vs_price(df_llm, save_path=OUTPUTS_FIGS / '17_llm_scores_vs_price.png')
plot_llm_coverage_by_zip(df_llm, save_path=OUTPUTS_FIGS / '18_llm_coverage_by_zip.png')
plt.show()

## LLM Score Inter-Correlations


In [ ]:
from src.plotting import (
    plot_llm_correlation_matrix,
    plot_llm_vs_structured_correlation,
    plot_luxury_vs_hardfacts,
)

# 1. LLM feature inter-correlation
_ = plot_llm_correlation_matrix(df_llm)

# 2. LLM vs structured cross-correlation
_ = plot_llm_vs_structured_correlation(df_llm)

# 3. Luxury score vs hard distress flags
_ = plot_luxury_vs_hardfacts(df_llm)

In [ ]:
import itertools
from src.features_structured import LLM_SCORE_COLS

corr_matrix = df_llm[LLM_SCORE_COLS].corr()
pairs = []
for a, b in itertools.combinations(LLM_SCORE_COLS, 2):
    pairs.append({
        'Feature A': a.replace('llm_', ''),
        'Feature B': b.replace('llm_', ''),
        'r': corr_matrix.loc[a, b]
    })
pair_df = pd.DataFrame(pairs).sort_values('r', key=abs, ascending=False)
print('Top 10 LLM score correlations (by |r|):')
display(pair_df.head(10).style.format({'r': '{:.3f}'}).set_caption('Pairwise Pearson Correlations'))

---
## 7 · Manual Audit — 20 Sampled Entries


In [ ]:
AUDIT_SEED = 42
AUDIT_N    = 20

# Sample from listings that have both a description and LLM output
has_llm  = df_llm['llm_luxury_score'].notna()
has_desc = df_llm['has_description']
audit_df = df_llm[has_llm & has_desc].sample(n=min(AUDIT_N, has_llm.sum()), random_state=AUDIT_SEED)

soft_cols = ['llm_luxury_score', 'llm_uniqueness_score', 'llm_renovation_quality_score',
             'llm_curb_appeal_score', 'llm_spaciousness_score',
             'llm_has_premium_finishes', 'llm_is_recently_updated', 'llm_soft_evidence']

hard_cols = ['llm_as_is_flag', 'llm_fixer_upper_flag', 'llm_needs_repair_flag',
             'llm_foundation_issue_flag', 'llm_roof_issue_flag', 'llm_water_damage_flag',
             'llm_foreclosure_flag', 'llm_cash_only_flag', 'llm_investor_special_flag',
             'llm_hard_evidence']

id_cols   = ['zpid', 'price', 'address_full']

print(f'Manual audit sample: {len(audit_df)} listings (seed={AUDIT_SEED})\n')
print('=' * 100)

for _, row in audit_df.reset_index(drop=True).iterrows():
    print(f"\n{'â”€'*100}")
    print(f"zpid: {row.get('zpid')}  |  Price: ${row.get('price', 0):,.0f}  |  {row.get('address_full', '')}")
    print()
    desc = str(row.get('property_description', ''))
    print('DESCRIPTION:', desc[:400] + ('...' if len(desc) > 400 else ''))
    print()
    print('SOFT FEATURES:')
    for col in soft_cols:
        val = row.get(col)
        if val is not None and str(val) not in ('nan', 'None', ''):
            print(f'  {col.replace("llm_",""):35s}: {val}')
    print('HARD FLAGS (True only):')
    hard_true = [c.replace('llm_','').replace('_flag','') for c in hard_cols
                 if c != 'llm_hard_evidence' and row.get(c) == 1]
    if hard_true:
        print('  ' + ', '.join(hard_true))
        evidence = row.get('llm_hard_evidence')
        if evidence:
            print(f'  Evidence: {evidence}')
    else:
        print('  (none)')

---
## 8 · Augmented Models

We estimate the full 7-model roster and compare the key structured-versus-augmented pairs:

| Model | Features |
|---|---|
| OLS-Structured | Lean structured baseline |
| OLS-Augmented | Lean structured baseline + LLM features |
| XGB-Structured | Lean structured baseline |
| XGB-Augmented | Lean structured baseline + LLM features |
| LassoCV-Structured | Lean structured baseline |
| LassoCV-Augmented | Lean structured baseline + LLM features |
| OLS-Lasso-Selected | Lasso-selected subset of the augmented feature space |


In [ ]:
from src.features_structured import build_feature_matrix, get_target
from src.modeling import (
    fit_ols, fit_xgboost, fit_lasso_cv,
    lasso_selected_features, shared_holdout_split,
    paired_rmse_test, build_two_panel_table,
)

y_llm = get_target(df_llm)

X_struct = build_feature_matrix(df_llm, include_llm=False)
X_augm   = build_feature_matrix(df_llm, include_llm=True)

# Shared hold-out on augmented sample (same seed = reproducible)
X_struct_train, X_struct_test, y_llm_train, y_llm_test = shared_holdout_split(X_struct, y_llm)
X_augm_train = X_augm.loc[X_struct_train.index]
X_augm_test  = X_augm.loc[X_struct_test.index]

print(f'Augmented sample hold-out: {len(X_struct_train):,} train / {len(X_struct_test):,} test')
print(f'Structured features: {X_struct.shape[1]}, Augmented features: {X_augm.shape[1]}')


In [ ]:
# OLS-Structured on the merged sample for apples-to-apples comparison
ols_s_same, metrics_s_same, _ = fit_ols(
    X_struct_train, y_llm_train,
    model_name='OLS-Structured',
    X_test=X_struct_test, y_test=y_llm_test,
)

# OLS-Augmented: structured baseline + LLM block
ols_l, metrics_l, coef_l = fit_ols(
    X_augm_train, y_llm_train,
    model_name='OLS-Augmented',
    X_test=X_augm_test, y_test=y_llm_test,
)
coef_l.to_csv(OUTPUTS_TABLES / 'coef_ols_augmented.csv')

# LLM coefficients
llm_coefs = coef_l[coef_l.index.str.startswith('llm_')].sort_values('p_value')
print('LLM feature coefficients (OLS-Augmented):')
display(llm_coefs)


In [ ]:
xgb_s, metrics_xgb_s = fit_xgboost(
    X_struct_train, y_llm_train,
    model_name='XGB-Structured',
    X_test=X_struct_test, y_test=y_llm_test,
)
xgb_l, metrics_xgb_l = fit_xgboost(
    X_augm_train, y_llm_train,
    model_name='XGB-Augmented',
    X_test=X_augm_test, y_test=y_llm_test,
)


In [ ]:
# Lasso: structured-only and augmented variants
lasso_s, metrics_lasso_s, lasso_coef_s = fit_lasso_cv(
    X_struct_train, y_llm_train,
    model_name='LassoCV-Structured',
    X_test=X_struct_test, y_test=y_llm_test,
)
lasso_a, metrics_lasso_a, lasso_coef_a = fit_lasso_cv(
    X_augm_train, y_llm_train,
    model_name='LassoCV-Augmented',
    X_test=X_augm_test, y_test=y_llm_test,
)

# Lasso-selected OLS: fit OLS using only features with non-zero Lasso coefficients
selected_features = lasso_selected_features(lasso_coef_a)
X_lsel_train = X_augm_train[selected_features]
X_lsel_test  = X_augm_test[selected_features]
print(f'Lasso-selected OLS uses {len(selected_features)} features (vs {X_augm.shape[1]} full)')

ols_lsel, metrics_lsel, coef_lsel = fit_ols(
    X_lsel_train, y_llm_train,
    model_name='OLS-Lasso-Selected',
    X_test=X_lsel_test, y_test=y_llm_test,
)

# Two-panel comparison table
all_metrics = [
    metrics_s_same, metrics_l,
    metrics_xgb_s, metrics_xgb_l,
    metrics_lasso_s, metrics_lasso_a,
    metrics_lsel,
]

panel_a, panel_b = build_two_panel_table(all_metrics)

print() 
print('Panel A: Test-set performance (shared 20% hold-out)')
display(panel_a.style.set_caption('Panel A: Test-Set Metrics'))
print()
print('Panel B: Cross-validated performance (5x5 repeated K-fold on 80% training set)')
display(panel_b.style.set_caption('Panel B: CV Metrics (mean +/- SD)'))

# Save both panels
panel_a.to_csv(OUTPUTS_TABLES / 'comparison_panel_a_test.csv', index=False)
panel_b.to_csv(OUTPUTS_TABLES / 'comparison_panel_b_cv.csv', index=False)

# Paired t-test: is OLS-Augmented significantly better than OLS-Structured out-of-sample?
ttest_result = paired_rmse_test(
    metrics_s_same['fold_rmses'], metrics_l['fold_rmses'],
    model_a_name='OLS-Structured',
    model_b_name='OLS-Augmented',
)
pd.DataFrame([ttest_result]).to_csv(OUTPUTS_TABLES / 'paired_ttest_ols_structured_vs_augmented.csv', index=False)

print()
print('Paired t-test on fold-level RMSE (H0: RMSE_OLS-Structured = RMSE_OLS-Augmented):')
print(f"  Mean RMSE OLS-Structured: ${ttest_result['mean_rmse_a']:,.0f}")
print(f"  Mean RMSE OLS-Augmented: ${ttest_result['mean_rmse_b']:,.0f}")
print(f"  Difference:      ${ttest_result['rmse_diff']:,.0f}")
print(f"  t-statistic:     {ttest_result['t_statistic']:.3f}")
print(f"  p-value:         {ttest_result['p_value']:.4f}")
print(f"  Significant at 5%: {ttest_result['significant_005']}")


## Multicollinearity Diagnostics



In [ ]:
from src.modeling import compute_vif

# Compute VIF for the augmented feature matrix
vif_df = compute_vif(X_augm)

# Show features with VIF > 5 (concerning multicollinearity)
print(f'Features with VIF > 5: {vif_df["collinear"].sum()} of {len(vif_df)}')
print(f'Features with VIF > 10: {(vif_df["VIF"] > 10).sum()} of {len(vif_df)}')
print()

# Display top 20 features by VIF
display(vif_df.head(20).style.set_caption('Top 20 features by VIF (augmented model)'))

# Show LLM-specific VIF values
llm_vif = vif_df[vif_df['feature'].str.startswith('llm_')].reset_index(drop=True)
print(f'\nLLM features with VIF > 5:')
display(llm_vif[llm_vif['collinear']])


## Per-Feature Predictive Power: Marginal R-squared Contribution




In [ ]:
from src.modeling import marginal_feature_contribution
from src.features_structured import LLM_SCORE_COLS, LLM_FLAG_COLS

# Test all LLM features individually against the OLS-Structured baseline
test_cols = [c for c in LLM_SCORE_COLS + LLM_FLAG_COLS if c in df_llm.columns]
marginal_df = marginal_feature_contribution(
    X_struct, y_llm, test_cols, metrics_s_same, source_df=df_llm
)
print(f'Marginal R-squared gain for {len(test_cols)} LLM features:')
display(marginal_df.round(4))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
marginal_df_sorted = marginal_df.sort_values('R2_gain')
colors = ['#2196F3' if g > 0 else '#90CAF9' for g in marginal_df_sorted['R2_gain']]
ax.barh(
    marginal_df_sorted['feature'].str.replace('llm_', ''),
    marginal_df_sorted['R2_gain'],
    color=colors
)
ax.set_xlabel('Marginal R-squared Gain over OLS-Structured Baseline')
ax.set_title('Individual LLM Feature Predictive Power')
ax.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.savefig('../outputs/figures/32_marginal_r2_gain.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: outputs/figures/32_marginal_r2_gain.png')


## Linearity F-Test for LLM Scores

In [ ]:
from src.features_structured import LLM_SCORE_COLS, TARGET_COL
from src.modeling import linearity_f_test_scores

linearity_df = linearity_f_test_scores(df_llm[TARGET_COL], df_llm[LLM_SCORE_COLS])
linearity_df.to_csv(OUTPUTS_TABLES / "f_test_linearity.csv", index=False)
display(linearity_df.style.set_caption("F-test for linearity of LLM scores"))


## Formal Significance Test: Partial F-Test



In [ ]:
from src.modeling import partial_f_test

# Partial F-test: OLS-Structured (restricted) vs OLS-Augmented (full)
# Both models are fitted on the same LLM-merged sample (ols_s_same from cell above)
f_result = partial_f_test(ols_s_same, ols_l)

print('Partial F-test: OLS-Structured (structured) vs OLS-Augmented (augmented)')
print('H0: All LLM feature coefficients are jointly zero')
print()
display(f_result)

# Extract key statistics
f_stat = f_result.iloc[1]['F']
p_val = f_result.iloc[1]['Pr(>F)']
df_diff = int(f_result.iloc[1]['df_diff'])
print(f'\nF-statistic: {f_stat:.2f}')
print(f'p-value: {p_val:.2e}')
print(f'Additional parameters (LLM features): {df_diff}')
print(f'\nConclusion: {"REJECT H0" if p_val < 0.01 else "FAIL TO REJECT H0"} at p < 0.01')


## Robustness Check: Controlling for Description Quality


In [ ]:
# Robustness: does LLM improvement survive controlling for description quality?

# Derive word count if not already present
if 'description_word_count' not in df_llm.columns:
    df_llm['description_word_count'] = df_llm['property_description'].fillna('').str.split().str.len()

# OLS-Structured + description quality controls
X_dq = build_feature_matrix(
    df_llm, include_llm=False,
    extra_numeric=['llm_description_quality', 'description_word_count']
)
ols_s_dq, metrics_s_dq, _ = fit_ols(
    X_dq, y_llm, model_name='OLS-Structured+DQ (desc. quality controls)', run_cv=True
)

# Compare: OLS-Structured baseline vs OLS-Structured+DQ vs OLS-Augmented
print('Endogeneity Robustness Check:')
print(f"  OLS-Structured baseline (structured only):   R² = {metrics_s_same['R²']:.4f}  |  CV R² = {metrics_s_same.get('CV_R²', 'N/A')}")
print(f"  OLS-Structured + description quality:        R² = {metrics_s_dq['R²']:.4f}  |  CV R² = {metrics_s_dq.get('CV_R²', 'N/A')}")
print(f"  OLS-Augmented augmented (all LLM features): R² = {metrics_l['R²']:.4f}  |  CV R² = {metrics_l.get('CV_R²', 'N/A')}")
print(f"\n  R² gain from DQ controls alone:  {metrics_s_dq['R²'] - metrics_s_same['R²']:.4f}")
print(f"  R² gain from full LLM features:  {metrics_l['R²'] - metrics_s_same['R²']:.4f}")
print(f"  R² gain after controlling DQ:    {metrics_l['R²'] - metrics_s_dq['R²']:.4f}")

if metrics_l['R²'] > metrics_s_dq['R²']:
    print('\n  --> LLM features capture information BEYOND mere description quality.')
else:
    print('\n  --> LLM improvement is largely explained by description quality proxy.')


## 8b – Variable Selection & Dimensionality Reduction



In [ ]:
from src.plotting import plot_lasso_coefficients

# Use augmented Lasso already fitted in the comparison section
lasso_coef_a.attrs['alpha'] = metrics_lasso_a['alpha']

# Visualise
_ = plot_lasso_coefficients(lasso_coef_a, model_name='LassoCV (augmented)')

print(f"LassoCV selected {lasso_coef_a['selected'].sum()}/{len(lasso_coef_a)} features at alpha={metrics_lasso_a['alpha']:.6f}")


In [ ]:
# Visualise Lasso-selected features (structured-only)
_ = plot_lasso_coefficients(lasso_coef_s, model_name='LassoCV (structured)')


In [ ]:
# OLS-Lasso-Selected already fitted in the comparison section above
print(f'OLS-Lasso-Selected uses {len(selected_features)} features (vs {X_augm.shape[1]} in OLS-Augmented)')
coef_lsel.to_csv(OUTPUTS_TABLES / 'coef_ols_lasso_selected.csv')

# Show key coefficients
non_fe_lsel = coef_lsel[~coef_lsel.index.str.startswith('zip_code_')].copy()
display(non_fe_lsel.head(15))


## Dimensionality Reduction: PCA on LLM Scores


In [ ]:
from src.modeling import run_pca_scores
from src.plotting import plot_pca_scree, plot_pca_loadings, plot_pca_scatter

# PCA on the 5 core LLM quality scores
pca_score_cols = [
    'llm_luxury_score', 'llm_uniqueness_score',
    'llm_renovation_quality_score', 'llm_curb_appeal_score',
    'llm_spaciousness_score',
]
pca, pc_scores, pca_scaler = run_pca_scores(df_llm, score_cols=pca_score_cols)

# Scree plot and loadings
_ = plot_pca_scree(pca, title_suffix='LLM Quality Scores')
_ = plot_pca_loadings(pca, feature_names=pca_score_cols)

# 2D scatter: PC1 vs PC2, colored by price
_ = plot_pca_scatter(pc_scores, price=df_llm.loc[pc_scores.index, 'log_price'])

# Print component interpretation
print("\n--- PCA Component Interpretation ---")
loadings = pd.DataFrame(
    pca.components_.T,
    index=[c.replace('llm_', '').replace('_score', '') for c in pca_score_cols],
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
)
for pc in loadings.columns:
    top = loadings[pc].abs().nlargest(3)
    signs = [f"{loadings.loc[f, pc]:+.2f} {f}" for f in top.index]
    var_pct = pca.explained_variance_ratio_[int(pc[2:])-1]
    print(f"  {pc} ({var_pct:.1%} variance): {', '.join(signs)}")

## Property Segments: K-Means Clustering on LLM Profiles



In [ ]:
from src.modeling import cluster_listings
from src.plotting import plot_cluster_elbow, plot_cluster_profiles, plot_cluster_price_box

# Elbow method to choose k
_, cluster_labels, inertia_df = cluster_listings(pc_scores, n_clusters=4, max_k=8)
_ = plot_cluster_elbow(inertia_df)

# Cluster profiles: structural + LLM features per cluster
_ = plot_cluster_profiles(df_llm.loc[pc_scores.index], cluster_labels)

# Price distribution per cluster — the economic payoff
_ = plot_cluster_price_box(df_llm.loc[pc_scores.index], cluster_labels)

# Summary table: cluster means for key variables
profile_cols = [
    'price', 'living_area_sqft', 'bedrooms', 'bathrooms', 'property_age',
    'llm_luxury_score', 'llm_renovation_quality_score', 'llm_spaciousness_score',
]
avail = [c for c in profile_cols if c in df_llm.columns]
cluster_summary = df_llm.loc[pc_scores.index, avail].copy()
cluster_summary['cluster'] = cluster_labels.values
summary = cluster_summary.groupby('cluster')[avail].agg(['mean', 'median', 'count'])
print("\n--- Cluster Summary (mean values) ---")
display(cluster_summary.groupby('cluster')[avail].mean().round(1).style.set_caption(
    'Mean Structural & LLM Features by Cluster'))

In [ ]:
# Cluster-level statistics: median price and mean LLM scores per segment
cluster_stats = df_llm.loc[pc_scores.index].copy()
cluster_stats['cluster'] = cluster_labels.values
cluster_stats = cluster_stats.groupby('cluster').agg(
    n=('log_price', 'count'),
    median_price=('price', 'median'),
    mean_luxury=('llm_luxury_score', 'mean'),
    mean_renovation=('llm_renovation_quality_score', 'mean'),
    distress_rate=('llm_foreclosure_flag', 'mean'),
).round(2)
display(cluster_stats.style.set_caption(
    'Cluster Statistics: Median Price and Mean LLM Scores by Segment'))

---
## 9 · Interpretation and Diagnostics

In [ ]:
from src.modeling import get_feature_importance
from src.plotting import plot_feature_importance

# XGB-Augmented importance
imp = get_feature_importance(xgb_l, X_augm.columns.tolist())
_ = plot_feature_importance(imp, top_n=25, title='XGB-Augmented: Top-25 Features (gain)',
                             save_name='xgb_augmented_importance')
plt.show()
imp.head(25).to_csv(OUTPUTS_TABLES / 'xgb_augmented_importance.csv', index=False)

# XGB-Structured importance
imp_s = get_feature_importance(xgb_s, X_struct.columns.tolist())
_ = plot_feature_importance(imp_s, top_n=20, title='XGB-Structured: Top-20 Features (gain)',
                             save_name='xgb_structured_importance')
plt.show()

In [ ]:
from src.plotting import plot_shap_summary

# SHAP requires shap package: pip install shap
_ = plot_shap_summary(xgb_l, X_augm, save_name='shap_xgb_augmented')
plt.show()

In [ ]:
from src.plotting import plot_luxury_vs_ppsf_zip
from src.plotting_3d import chart_d_clusters, save_deck, show_deck

# Luxury vs $/sqft bubble
plot_luxury_vs_ppsf_zip(df_llm, save_path=OUTPUTS_FIGS / '19_luxury_vs_ppsf_zip.png')

# K-means cluster map
deck_d = chart_d_clusters(df_llm)
save_deck(deck_d, OUTPUTS_FIGS / '3d_d_cluster_map.html')
show_deck(deck_d)

In [ ]:
# Residual map for the lean structured baseline (OLS-Structured)
from src.plotting import plot_residual_map_interactive
import statsmodels.api as sm

X_struct_c = sm.add_constant(X_struct, has_constant='add')
plot_residual_map_interactive(
    df_llm,
    ols_s_same.predict(X_struct_c),
    save_path=OUTPUTS_FIGS / '3d_f_residual_map.html',
)


---
## 10 · Save Outputs

In [ ]:
# Final merged dataset (processed)
df_llm.to_parquet(DATA_PROCESSED / 'listings_with_llm.parquet', index=False)
df_llm.to_csv(DATA_PROCESSED / 'listings_with_llm.csv', index=False)

# Clean dataset (no LLM features) for replication
df_clean.to_parquet(DATA_PROCESSED / 'listings_clean.parquet', index=False)

# Model comparison (two-panel format)
panel_a_export = panel_a.reset_index()
panel_b_export = panel_b.reset_index()
panel_a_export.to_csv(OUTPUTS_TABLES / 'comparison_panel_a_test.csv', index=False)
panel_b_export.to_csv(OUTPUTS_TABLES / 'comparison_panel_b_cv.csv', index=False)
pd.DataFrame([ttest_result]).to_csv(OUTPUTS_TABLES / 'paired_ttest_ols_structured_vs_augmented.csv', index=False)

# Coefficient tables
coef_s.to_csv(OUTPUTS_TABLES / 'coef_ols_structured.csv')
coef_l.to_csv(OUTPUTS_TABLES / 'coef_ols_augmented.csv')
coef_lsel.to_csv(OUTPUTS_TABLES / 'coef_ols_lasso_selected.csv')

# Compact 4-model table used by the app's legacy comparison view
model_comparison = (
    pd.DataFrame(all_metrics)
    .rename(columns={'model': 'Model'})
    .loc[lambda d: d['Model'].isin(['OLS-Structured', 'OLS-Augmented', 'XGB-Structured', 'XGB-Augmented']),
         ['Model', 'R²', 'RMSE', 'MAE', 'MAPE (%)', 'CV_R²', 'CV_RMSE']]
)
model_comparison.to_csv(OUTPUTS_TABLES / 'model_comparison.csv', index=False)

# Feature importance
imp.to_csv(OUTPUTS_TABLES / 'xgb_augmented_importance.csv', index=False)

# Remove legacy OLS artefacts so the repo carries one naming scheme only
keep_tables = {
    'coef_ols_structured.csv',
    'coef_ols_augmented.csv',
    'coef_ols_lasso_selected.csv',
    'paired_ttest_ols_structured_vs_augmented.csv',
}
for legacy in OUTPUTS_TABLES.glob('coef_ols_*.csv'):
    if legacy.name not in keep_tables:
        legacy.unlink()
for legacy in OUTPUTS_TABLES.glob('paired_ttest_ols*.csv'):
    if legacy.name not in keep_tables:
        legacy.unlink()

keep_figures = {
    'ols_structured_actual_vs_predicted.png',
    '3d_f_residual_map.html',
}
for legacy in OUTPUTS_FIGS.glob('ols_*actual_vs_predicted.png'):
    if legacy.name not in keep_figures:
        legacy.unlink()
for legacy in OUTPUTS_FIGS.glob('18_residual_map_ols-*.png'):
    legacy.unlink()

# Summary
print('Saved files:')
for p in sorted(list(DATA_PROCESSED.glob('*')) + list(OUTPUTS_FIGS.glob('*')) + list(OUTPUTS_TABLES.glob('*'))):
    print(f'  {p.relative_to(root)}')


## Appendix: Partial F-Test (SSR Formula) and SHAP Analysis



### Partial F-Test from SSR (Training Set)




In [ ]:
# Partial F-test via SSR formula on the TRAINING set
from src.modeling import partial_f_test_from_ssr_models
import statsmodels.api as sm

# Restricted: OLS-Structured (no LLM features), fitted on training data
Xr_c = sm.add_constant(X_struct_train, has_constant='add')
ols_structured_train = sm.OLS(y_llm_train, Xr_c).fit()

# Unrestricted: OLS-Augmented (structured + 30 LLM features), fitted on training data
Xu_c = sm.add_constant(X_augm_train, has_constant='add')
ols_augmented_train = sm.OLS(y_llm_train, Xu_c).fit()

n_train = len(y_llm_train)
k_full = Xu_c.shape[1]              # includes intercept
q = k_full - Xr_c.shape[1]          # number of LLM features added (30)

f_test_row = partial_f_test_from_ssr_models(
    ols_restricted=ols_structured_train,
    ols_full=ols_augmented_train,
    n_train=n_train,
    k_full=k_full,
    q=q,
)
f_test_df = pd.DataFrame([f_test_row])
f_test_df.to_csv(OUTPUTS_TABLES / 'partial_f_test.csv', index=False)

print('Partial F-test (H0: all 30 LLM coefficients = 0)')
print(f"  SSR_structured : {f_test_row['SSR_structured']:.6f}")
print(f"  SSR_augmented  : {f_test_row['SSR_augmented']:.6f}")
print(f"  q (df1)        : {f_test_row['df1']}")
print(f"  n - k_full     : {f_test_row['df2']}")
print(f"  n_train        : {f_test_row['n_train']}")
print(f"  k_full         : {f_test_row['k_full']}")
print(f"  F-statistic    : {f_test_row['F_stat']:.4f}")
print(f"  p-value        : {f_test_row['p_value']:.3e}")
display(f_test_df)


### SHAP Analysis for XGB-Augmented


In [ ]:
# SHAP analysis for XGB-Augmented on the test set
from src.modeling import compute_shap_xgb, rank_shap_features

shap_values, base_values, shap_feature_names = compute_shap_xgb(xgb_l, X_augm_test)

# Persist raw SHAP arrays for reproducibility
np.save(DATA_PROCESSED / 'shap_values.npy', shap_values)
np.save(DATA_PROCESSED / 'shap_base_values.npy', base_values)
np.save(DATA_PROCESSED / 'shap_feature_names.npy', np.array(shap_feature_names, dtype=object))

# Mean |SHAP| ranking
shap_ranking = rank_shap_features(shap_values, shap_feature_names)
shap_ranking.to_csv(OUTPUTS_TABLES / 'shap_feature_ranking.csv', index=False)

print('Top 10 features by mean |SHAP| (XGB-Augmented, test set):')
display(shap_ranking.head(10))


In [ ]:
# SHAP summary plot (beeswarm, top 20)
import shap

plt.figure()
shap.summary_plot(shap_values, X_augm_test, show=False, max_display=20, plot_type='dot')
fig = plt.gcf()
fig.savefig(OUTPUTS_FIGS / 'shap_summary_xgb_augmented.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# SHAP bar plot (mean |SHAP|, top 20)
plt.figure()
shap.summary_plot(shap_values, X_augm_test, show=False, max_display=20, plot_type='bar')
fig = plt.gcf()
fig.savefig(OUTPUTS_FIGS / 'shap_bar_xgb_augmented.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)


In [ ]:
# SHAP dependence plots for top 5 features by mean |SHAP|
top5 = shap_ranking['feature'].head(5).tolist()
print('Dependence plots for:', top5)

for feat in top5:
    safe = feat.replace('/', '_').replace(' ', '_')
    plt.figure()
    shap.dependence_plot(feat, shap_values, X_augm_test, show=False, interaction_index='auto')
    fig = plt.gcf()
    fig.savefig(OUTPUTS_FIGS / f'shap_dependence_{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
